# 06 — LlamaIndex Integration: Hybrid Retrieval with SimlarRetriever

`SimlarRetriever` is a LlamaIndex `BaseRetriever` backed by simlar's `HelixIndex`.
It slots into any LlamaIndex pipeline — `RetrieverQueryEngine`, `CondenseQuestionChatEngine`, routers, and more — while running simlar under the hood

**What we cover**
- Embedding a corpus with LlamaIndex's `HuggingFaceEmbedding`
- Building a `SimlarRetriever` with the `from_texts()` convenience factory
- Running queries — inspecting `NodeWithScore` results
- Building manually: constructing a `HelixIndex` and wiring it to the retriever
- Saving to disk with `persist()` and reloading with `from_persist()`
- Wiring into a `RetrieverQueryEngine` for RAG

In [ ]:
%pip install -q datasets llama-index-core llama-index-embeddings-huggingface simlar

## Load the dataset

We use [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — 120,000 news headlines across four categories.
We index 500 articles and keep a side-table (`id_to_meta`) for category labels; the retriever stores text only.

In [ ]:
from datasets import load_dataset

LABEL_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

ds     = load_dataset("fancyzhx/ag_news", split="train[:500]")
texts  = ds["text"]
labels = [LABEL_NAMES[l] for l in ds["label"]]
ids    = [str(i) for i in range(len(texts))]

id_to_meta = {str(i): {"category": labels[i]} for i in range(len(texts))}

print(f"Loaded {len(texts)} articles")
print(f"Category distribution: { {k: labels.count(k) for k in LABEL_NAMES.values()} }")
print(f"\nSample: {texts[0][:120]}")

## Embed the corpus

`SimlarRetriever` is embedding-agnostic — pass any LlamaIndex `BaseEmbedding`. The same model must be used for both indexing and querying.

Here we use `HuggingFaceEmbedding` with `BAAI/bge-small-en-v1.5` (384-dim, runs on CPU).

In [ ]:
import numpy as np
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

vectors = np.array(
    embed_model.get_text_embedding_batch(texts, show_progress=True),
    dtype=np.float32,
)

print(f"Embedded {len(texts)} documents  |  dim: {vectors.shape[1]}")

## Build the retriever

`SimlarRetriever.from_texts()` creates a `HelixIndex` internally and wires everything together in one call.

In [ ]:
from simlar.integrations.llama_index.simlar_retriever import SimlarRetriever

retriever = SimlarRetriever.from_texts(
    texts=texts,
    ids=ids,
    vectors=vectors,
    embed_model=embed_model,
    k=5,
)

index = retriever.client  # underlying HelixIndex
print(f"Index size  : {index.size}")
print(f"Is trained  : {index.is_trained}")
print(f"Index type  : {index.index_type}")

## Run queries

`retriever.retrieve(query_str)` returns a list of `NodeWithScore` objects.
Each node's `.text` holds the document text; `.score` is the RRF-fused ranking score.
We look up the news category from `id_to_meta` for display.

In [ ]:
def run_query(question: str) -> None:
    nodes = retriever.retrieve(question)
    print(f"Query : '{question}'")
    print(f"Results ({len(nodes)}):")
    for n in nodes:
        doc_id = n.node.id_
        cat    = id_to_meta[doc_id]["category"]
        rank   = n.node.metadata.get("rank", "?")
        print(f"  [{cat:8s}]  rank={rank}  score={n.score:.4f}  {n.node.text[:90]}")
    print()


run_query("technology startup funding Silicon Valley")
run_query("Olympic Games world record athlete")
run_query("interest rate Federal Reserve inflation")

## NodeWithScore anatomy

Every result is a standard LlamaIndex `NodeWithScore`. The inner `TextNode` carries:

| Attribute | Value |
|---|---|
| `node.id_` | The document ID passed at index time |
| `node.text` | The document text |
| `node.metadata["rank"]` | 1-based position in the result list |
| `node.metadata["score"]` | RRF-fused score (same as `n.score`) |
| `score` | Shorthand for `node.metadata["score"]` |

In [ ]:
example_nodes = retriever.retrieve("artificial intelligence machine learning")
top = example_nodes[0]

print(f"type        : {type(top)}")
print(f"score       : {top.score:.4f}")
print(f"node type   : {type(top.node).__name__}")
print(f"node.id_    : {top.node.id_}")
print(f"node.text   : {top.node.text[:120]}")
print(f"node.meta   : {top.node.metadata}")

## Manual construction

For finer control over `HelixIndex` parameters — candidate pool sizes, incremental adds — build the index yourself and pass it in.

In [ ]:
from simlar import HelixIndex

# Build in two stages: seed the index, then add more documents
helix = HelixIndex(text_k=500, vector_k=200, top_k=100)
helix.add(ids=ids[:300], texts=texts[:300], vectors=vectors[:300])
helix.add(ids=ids[300:], texts=texts[300:], vectors=vectors[300:])

manual_retriever = SimlarRetriever(
    index=helix,
    id_to_text=dict(zip(ids, texts)),
    embed_model=embed_model,
    k=5,
)

print(f"Manual index size: {manual_retriever.client.size}")

# Verify it returns results
nodes = manual_retriever.retrieve("space exploration NASA")
print(f"Top result  : {nodes[0].node.text[:120]}")

## Persist and reload

`persist(directory)` saves the `HelixIndex` and the `id_to_text` map to disk.
`SimlarRetriever.from_persist(directory, embed_model)` reconstructs the retriever without re-embedding.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    save_dir = str(Path(tmp) / "ag_news_retriever")

    # Persist
    retriever.persist(save_dir)
    files = [p.name for p in Path(save_dir).iterdir()]
    print(f"Saved to {save_dir}")
    print(f"Files: {files}")

    # Reload — no re-embedding needed
    reloaded = SimlarRetriever.from_persist(
        directory=save_dir,
        embed_model=embed_model,
        k=5,
    )
    print(f"\nReloaded index size: {reloaded.client.size}")

    # Verify results match
    question = "Federal Reserve interest rate decision"
    orig_ids     = [n.node.id_ for n in retriever.retrieve(question)]
    reloaded_ids = [n.node.id_ for n in reloaded.retrieve(question)]
    print(f"Results match: {orig_ids == reloaded_ids}")
    print(f"Top result: {reloaded.retrieve(question)[0].node.text[:120]}")

## Wire into a RetrieverQueryEngine

`SimlarRetriever` is a standard LlamaIndex `BaseRetriever`, so it plugs directly into `RetrieverQueryEngine` for RAG.

> **Note** — `RetrieverQueryEngine.query()` calls an LLM to synthesize a response. Configure your LLM via `llama_index.core.Settings.llm` before running the cell below. The retrieval step itself works without an LLM.

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

# --- configure your LLM here before querying ---
# from llama_index.llms.openai import OpenAI
# from llama_index.core import Settings
# Settings.llm = OpenAI(model="gpt-4o-mini")

synthesizer = get_response_synthesizer(response_mode="compact")
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
)

# Show retrieved context without an LLM
question = "What are recent developments in space technology?"
retrieved = retriever.retrieve(question)
print(f"Retrieved {len(retrieved)} nodes for: '{question}'")
for n in retrieved:
    cat = id_to_meta[n.node.id_]["category"]
    print(f"  [{cat:8s}]  score={n.score:.4f}  {n.node.text[:100]}")

print()
print("To get a synthesized answer, call:")
print("  response = query_engine.query(question)")
print("  print(response)")